# HAMIC — TPU downstream

Select a finished ModernBERT run, choose tasks and a protocol, then export the checkpoint and
fine-tune it. The checkpoint is exported to a local directory and imported back through
`import_weights` before fine-tuning, so a broken export fails here instead of silently changing
the model.

`quick` is one learning rate and one seed for smoke checks; `full` is the published HAMIC
protocol. This notebook never publishes. The token is read as `HF_KEY` (TPU only).

In [ ]:
HAMIC_REF = "main"
HUB_REPO = "HaianCao/hamic-runs"
DATA_REPO = "HaianCao/hamic-data"
XET_HIGH_PERFORMANCE = True

# Empty = first finished ModernBERT run. Use an exact run id from the discovery cell.
RUN_ID = ""
TASKS = ("vsmec",)
PROTOCOL = "quick"

In [ ]:
import json
import os
import shutil
import sys
import tarfile
import tempfile
import urllib.request
from pathlib import Path

os.environ["HF_XET_HIGH_PERFORMANCE"] = "1" if XET_HIGH_PERFORMANCE else "0"
work_root = Path(tempfile.gettempdir()) / "hamic-tpu"
source_dir = work_root / "source"
stamp = work_root / "source-ref.json"
repo = "https://github.com/HaianCao/HAMIC-LM"
with urllib.request.urlopen(
    f"https://api.github.com/repos/HaianCao/HAMIC-LM/commits/{HAMIC_REF}",
    timeout=30,
) as response:
    commit = json.load(response)["sha"]
stamp_data = json.loads(stamp.read_text()) if stamp.is_file() else None
if not isinstance(stamp_data, dict):
    stamp_data = None
if (
    not source_dir.is_dir()
    or stamp_data is None
    or stamp_data.get("ref") != HAMIC_REF
    or stamp_data.get("commit") != commit
):
    archive = work_root / "source.tar.gz"
    work_root.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(f"{repo}/archive/{commit}.tar.gz", archive)
    unpack = work_root / "unpack"
    shutil.rmtree(unpack, ignore_errors=True)
    unpack.mkdir()
    with tarfile.open(archive) as tar:
        try:
            tar.extractall(unpack, filter="data")
        except TypeError:
            tar.extractall(unpack)
    (top,) = unpack.iterdir()
    shutil.rmtree(source_dir, ignore_errors=True)
    shutil.move(str(top), source_dir)
    shutil.rmtree(unpack, ignore_errors=True)
    archive.unlink()
    stamp.write_text(json.dumps({"ref": HAMIC_REF, "commit": commit}))
sys.path.insert(0, str(source_dir / "src"))
if "hamic_jax" in sys.modules:
    raise RuntimeError(
        "An older hamic_jax is already imported. Restart the kernel and run this notebook again."
    )
import hamic_jax

if Path(hamic_jax.__file__).resolve().parent != (source_dir / "src" / "hamic_jax").resolve():
    raise RuntimeError("hamic_jax was imported from outside the pinned source tree")
print(f"Source {HAMIC_REF} resolved to {commit[:12]} · work root {work_root}")
print("The wheel excludes src/hamic_jax, so this notebook imports the pinned checkout.")

In [ ]:
from hamic_jax.dependencies import ensure_dependencies

installed = ensure_dependencies(downstream=True)
print(
    "Runtime and downstream dependencies are available."
    if not installed
    else f"Installed: {', '.join(installed)}"
)

In [ ]:
import jax

backend = jax.default_backend()
if backend != "tpu":
    raise RuntimeError(
        f"No TPU attached (jax.default_backend()={backend!r}). Attach the TPU and rerun."
    )
devices = jax.devices()
print(
    f"JAX {jax.__version__} · backend {backend} · {len(devices)} device(s) "
    f"{devices[0].device_kind} · code {commit[:12]}"
)

In [ ]:
import getpass

hub_token = os.environ.get("HF_KEY", "").strip()
if not hub_token:
    hub_token = getpass.getpass("Hugging Face token (HF_KEY): ").strip()
if not hub_token:
    raise RuntimeError("Enter a Hugging Face token in HF_KEY or at the prompt.")

In [ ]:
from hamic_jax.checkpoint import HuggingFaceObjectStore

run_store = HuggingFaceObjectStore(HUB_REPO, private=True, token=hub_token)
data_store = HuggingFaceObjectStore(DATA_REPO, repo_type="dataset", create=False, token=hub_token)
print(f"Hub account {run_store.whoami()} · runs {HUB_REPO} · data {DATA_REPO}")

In [ ]:
from hamic_jax.lab import complete_runs

encoders = [
    record
    for record in complete_runs(run_store)
    if record.manifest.get("preset", {}).get("architecture") == "modernbert"
]
if not encoders:
    raise RuntimeError("No finished ModernBERT run. Train an encoder on the TPU first.")
for record in encoders:
    print(
        f"{record.run_id} · step {record.step} · "
        f"{record.manifest.get('preset', {}).get('name', '')} · "
        f"{record.manifest.get('param_count')} params · {record.created}"
    )

In [ ]:
from IPython.display import display
from tqdm.auto import tqdm

from hamic_jax.lab import run_downstream
from hamic_jax.plots import downstream_bars

selected_id = RUN_ID.strip() or encoders[0].run_id
record = next((item for item in encoders if item.run_id == selected_id), None)
if record is None:
    raise KeyError(f"{selected_id!r} is not a finished ModernBERT run")
if not TASKS:
    raise ValueError("Select at least one task.")

with tqdm(total=len(TASKS), desc=f"Fine-tuning {record.run_id}", unit="task") as bar:
    def progress(index, total, name):
        bar.n = index
        bar.set_postfix_str(f"{name} · {index}/{total}")
        bar.refresh()

    report = run_downstream(
        record,
        run_store,
        data_store,
        work_root,
        source_dir=source_dir,
        token=hub_token,
        protocol=PROTOCOL,
        tasks=TASKS,
        progress=progress,
        log=print,
    )

rows = [{"task": name, **result} for name, result in report["results"].items()]
print(f"Report {report['report_path']}")
for row in rows:
    print(row)
display(downstream_bars(report["results"]))